# Running Models — Audio · Part 5 — Companion Notebook

This goes with **"How Audio Becomes Tokens: Neural Codecs and Encodec"**. Run cells top to bottom (Shift+Enter). This one is CPU-friendly — no GPU needed.

## Install

Encodec ships in `transformers`; `torchaudio` loads and resamples the clip.

In [ ]:
!pip install -q transformers torchaudio

## Load Encodec and a clip

The 24 kHz codec. Resample the clip to the codec's native rate before handing it over.

In [ ]:
from transformers import EncodecModel, AutoProcessor
import torchaudio

model = EncodecModel.from_pretrained("facebook/encodec_24khz")
processor = AutoProcessor.from_pretrained("facebook/encodec_24khz")

path = torchaudio.utils.download_asset(
    "tutorial-assets/Lab41-SRI-VOiCES-src-sp0307-ch127535-sg0042.wav"
)
waveform, sr = torchaudio.load(path)
waveform = torchaudio.functional.resample(waveform, sr, processor.sampling_rate)

inputs = processor(
    raw_audio=waveform[0].numpy(),
    sampling_rate=processor.sampling_rate,
    return_tensors="pt",
)

## Encode — the payoff

Push the clip through the encoder + quantizer. What comes back is **integers** from a fixed codebook — a vocabulary a transformer could model.

In [ ]:
encoded = model.encode(inputs["input_values"], inputs["padding_mask"])
codes = encoded.audio_codes
print(codes.shape)                 # [1, 1, n_codebooks, time] -> small integers
print(int(codes.min()), int(codes.max()))   # e.g. 0 .. 1023

Each frame of audio became a handful of integers (one per codebook, thanks to residual quantization), each in `[0, 1023]`. A second of audio is only a few thousand integers — a sequence, just like a sentence of word-tokens.

## Decode — and listen

Hand the codes (and the scales Encodec computed alongside them) to the decoder, then play the reconstruction.

In [ ]:
decoded = model.decode(encoded.audio_codes, encoded.audio_scales, inputs["padding_mask"])[0]
from IPython.display import Audio
Audio(decoded[0].detach().numpy(), rate=processor.sampling_rate)

It sounds (nearly) like the original — a codec is lossy compression, so don't expect bit-exact output. The codec learned to pack audio into tokens and unpack it.

**Copy vs. predict:** this was a round trip. Generation = train a transformer to *predict* these tokens instead of copying them, then run the predicted tokens through the decoder. That's MusicGen, next post.